# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_08 — Avellaneda-Stoikov Market Making

### Research question

How can we design, simulate, stress-test, and govern an inventory-aware market-making strategy using the Avellaneda-Stoikov framework?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
06_08_avellaneda_stoikov_market_making
```

The previous notebooks covered execution scheduling, dynamic execution, impact, Hawkes order flow, rough volatility, a C++ microstructure core, and futures constraints. This notebook studies a central model for quoting bid and ask prices under inventory risk.

It covers:

1. market making objective;
2. inventory risk;
3. CARA utility intuition;
4. reservation price;
5. optimal half-spread;
6. bid/ask quote skew;
7. order arrival intensity;
8. Poisson fills;
9. inventory dynamics;
10. cash and mark-to-market PnL;
11. baseline symmetric market maker;
12. Avellaneda-Stoikov market maker;
13. volatility-regime adaptation;
14. liquidity-regime adaptation;
15. inventory limits;
16. kill-switch controls;
17. adverse selection extension;
18. Monte Carlo PnL distribution;
19. parameter sensitivity;
20. quote diagnostics;
21. fill diagnostics;
22. risk metrics;
23. governance flags;
24. saved outputs and manifest.

The key idea:

> Avellaneda-Stoikov market making shifts quotes away from the mid-price according to inventory risk: when you are long, quote lower to encourage selling; when you are short, quote higher to encourage buying.

## 1. Market-making problem

A market maker posts:

$$
bid_t < S_t < ask_t
$$

and earns spread when buy and sell market orders hit their quotes.

But the market maker also accumulates inventory:

$$
q_t
$$

Inventory is risky because the mid-price moves.

A naive market maker posts symmetric quotes around the mid:

$$
bid_t = S_t - \delta
$$

$$
ask_t = S_t + \delta
$$

An inventory-aware market maker shifts quotes using a reservation price:

$$
r_t = S_t - q_t \gamma \sigma^2 (T-t)
$$

where:

- $S_t$: mid-price;
- $q_t$: inventory;
- $\gamma$: risk aversion;
- $\sigma$: volatility;
- $T-t$: remaining horizon.

If $q_t>0$, reservation price falls, encouraging inventory reduction.

If $q_t<0$, reservation price rises, encouraging inventory increase.

## 2. Avellaneda-Stoikov quotes

With exponential order-arrival intensity:

$$
\lambda(\delta)=A e^{-k\delta}
$$

the approximate optimal half-spread is:

$$
\begin{aligned}
\delta^* &= \frac{1}{\gamma}\log\Big(1+\frac{\gamma}{k}\Big) \\
&\quad + \frac{1}{2}\gamma\sigma^2(T-t)
\end{aligned}
$$

Then:

$$
bid_t = r_t - \delta^*
$$

$$
ask_t = r_t + \delta^*
$$

where:

$$
r_t = S_t - q_t\gamma\sigma^2(T-t)
$$

The half-spread has two components:

1. **liquidity component**: $\frac{1}{\gamma}\log(1+\gamma/k)$;
2. **risk component**: $\frac{1}{2}\gamma\sigma^2(T-t)$.

Higher risk aversion or volatility widens quotes.

Higher order-arrival sensitivity $k$ narrows quotes.

## 3. Fill probabilities

If bid and ask distances from mid are:

$$
\delta_b = S_t - bid_t
$$

$$
\delta_a = ask_t - S_t
$$

then the model assumes fill intensities:

$$
\lambda_b = A e^{-k\delta_b}
$$

$$
\lambda_a = A e^{-k\delta_a}
$$

For a small time step $dt$:

$$
P(\text{bid fill}) \approx 1-e^{-\lambda_b dt}
$$

$$
P(\text{ask fill}) \approx 1-e^{-\lambda_a dt}
$$

Bid fill means the market maker buys one unit and inventory increases.

Ask fill means the market maker sells one unit and inventory decreases.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class AvellanedaStoikovConfig:
    horizon_seconds: int = 3_600
    dt_seconds: float = 1.0
    seed: int = 42
    initial_mid: float = 100.0
    tick_size: float = 0.01
    order_size: float = 1.0
    initial_inventory: float = 0.0
    max_inventory: int = 20
    gamma: float = 0.08
    sigma_daily: float = 0.025
    trading_day_seconds: float = 23_400.0
    arrival_A: float = 1.60
    arrival_k: float = 1.40
    taker_adverse_selection_bps: float = 0.15
    min_spread_ticks: int = 1
    max_spread_ticks: int = 80
    inventory_kill_limit: int = 25
    drawdown_kill_limit: float = -750.0
    monte_carlo_paths: int = 2_000
    sensitivity_paths: int = 500
    output_subdir: str = "avellaneda_stoikov_market_making_v1"

config = AvellanedaStoikovConfig()
config

## 5. Helper functions

We need:

- tick rounding;
- reservation price;
- optimal half-spread;
- quote generation;
- fill intensities;
- PnL accounting.

In [ ]:
def round_down_to_tick(x, tick):
    return np.floor(x / tick) * tick

def round_up_to_tick(x, tick):
    return np.ceil(x / tick) * tick

def sigma_per_sqrt_second(config):
    return config.sigma_daily / np.sqrt(config.trading_day_seconds)

def remaining_time_fraction(step, n_steps):
    return max((n_steps - step) / n_steps, 0.0)

def reservation_price(mid, inventory, gamma, sigma_daily, tau_days):
    return mid - inventory * gamma * sigma_daily**2 * tau_days

def optimal_half_spread(gamma, sigma_daily, tau_days, k):
    if gamma <= 0:
        liquidity_component = 1.0 / max(k, 1e-12)
    else:
        liquidity_component = (1.0 / gamma) * np.log(1.0 + gamma / max(k, 1e-12))
    risk_component = 0.5 * gamma * sigma_daily**2 * tau_days
    return liquidity_component + risk_component

def make_quotes(mid, inventory, step, n_steps, config, strategy="AS"):
    tau_days = remaining_time_fraction(step, n_steps)

    if strategy == "symmetric":
        half = optimal_half_spread(config.gamma, config.sigma_daily, 0.0, config.arrival_k)
        r = mid
    elif strategy == "AS":
        r = reservation_price(mid, inventory, config.gamma, config.sigma_daily, tau_days)
        half = optimal_half_spread(config.gamma, config.sigma_daily, tau_days, config.arrival_k)
    else:
        raise ValueError("strategy must be symmetric or AS")

    half = np.clip(
        half,
        config.min_spread_ticks * config.tick_size / 2.0,
        config.max_spread_ticks * config.tick_size / 2.0,
    )

    raw_bid = r - half
    raw_ask = r + half

    bid = round_down_to_tick(raw_bid, config.tick_size)
    ask = round_up_to_tick(raw_ask, config.tick_size)

    if ask <= bid:
        ask = bid + config.tick_size

    return {
        "reservation_price": r,
        "half_spread": half,
        "bid": bid,
        "ask": ask,
        "bid_distance": mid - bid,
        "ask_distance": ask - mid,
        "spread": ask - bid,
    }

def fill_intensity(distance, A, k):
    return A * np.exp(-k * max(distance, 0.0))

def fill_probability(intensity, dt_seconds):
    return 1.0 - np.exp(-intensity * dt_seconds)

# Example.
make_quotes(100.0, inventory=10, step=100, n_steps=3600, config=config, strategy="AS")

## 6. Simulate a mid-price path

The baseline mid-price follows:

$$
dS_t = \sigma S_t dW_t
$$

The model is deliberately simple so the market-making logic is visible.

Later extensions can add jumps, Hawkes order flow, rough volatility, or microprice drift.

In [ ]:
def simulate_mid_price_path(config, seed=None, volatility_multiplier=1.0):
    if seed is None:
        seed = config.seed

    rng = np.random.default_rng(seed)
    n_steps = int(config.horizon_seconds / config.dt_seconds)
    sigma_sec = sigma_per_sqrt_second(config) * volatility_multiplier

    returns = sigma_sec * np.sqrt(config.dt_seconds) * rng.standard_normal(n_steps)
    mid = config.initial_mid * np.exp(np.cumsum(returns))
    mid = np.r_[config.initial_mid, mid]

    times = np.arange(n_steps + 1) * config.dt_seconds

    return pd.DataFrame({
        "step": np.arange(n_steps + 1),
        "time_seconds": times,
        "mid": mid,
        "return": np.r_[0.0, returns],
    })

mid_path = simulate_mid_price_path(config)

mid_path.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(mid_path["time_seconds"], mid_path["mid"])
plt.title("Simulated Mid-Price Path")
plt.xlabel("Time, seconds")
plt.ylabel("Mid price")
plt.show()

## 7. Single-path market-making simulator

At each step:

1. observe mid;
2. compute bid/ask quotes;
3. compute fill probabilities;
4. sample bid and ask fills;
5. update inventory and cash;
6. mark to market;
7. apply kill-switch checks.

Inventory update:

- bid fill: inventory $q \leftarrow q + 1$;
- ask fill: inventory $q \leftarrow q - 1$.

Cash update:

- bid fill: cash decreases by bid price;
- ask fill: cash increases by ask price.

In [ ]:
def run_market_making_path(config, strategy="AS", seed=None, volatility_multiplier=1.0, liquidity_multiplier=1.0, adverse_selection=True):
    if seed is None:
        seed = config.seed

    rng = np.random.default_rng(seed)
    mid_df = simulate_mid_price_path(config, seed=seed + 10, volatility_multiplier=volatility_multiplier)

    n_steps = len(mid_df) - 1
    inventory = float(config.initial_inventory)
    cash = 0.0
    killed = False
    kill_reason = "NONE"

    rows = []

    for step in range(n_steps):
        mid = float(mid_df.loc[step, "mid"])
        next_mid = float(mid_df.loc[step + 1, "mid"])

        quote = make_quotes(mid, inventory, step, n_steps, config, strategy=strategy)

        # Inventory limit: stop quoting side that worsens inventory.
        quote_bid_active = inventory < config.max_inventory
        quote_ask_active = inventory > -config.max_inventory

        lambda_bid = fill_intensity(quote["bid_distance"], config.arrival_A * liquidity_multiplier, config.arrival_k) if quote_bid_active else 0.0
        lambda_ask = fill_intensity(quote["ask_distance"], config.arrival_A * liquidity_multiplier, config.arrival_k) if quote_ask_active else 0.0

        p_bid = fill_probability(lambda_bid, config.dt_seconds)
        p_ask = fill_probability(lambda_ask, config.dt_seconds)

        bid_fill = rng.uniform() < p_bid
        ask_fill = rng.uniform() < p_ask

        # Adverse selection toy model: if filled, next mid slightly moves against us.
        adverse_move = 0.0
        if adverse_selection:
            adverse = config.taker_adverse_selection_bps / 10000.0 * mid
            if bid_fill:
                adverse_move -= adverse
            if ask_fill:
                adverse_move += adverse

        effective_next_mid = next_mid + adverse_move

        cash_before = cash
        inventory_before = inventory

        if bid_fill:
            inventory += config.order_size
            cash -= quote["bid"] * config.order_size

        if ask_fill:
            inventory -= config.order_size
            cash += quote["ask"] * config.order_size

        mtm = cash + inventory * effective_next_mid
        spread_capture = 0.0
        if bid_fill:
            spread_capture += (mid - quote["bid"]) * config.order_size
        if ask_fill:
            spread_capture += (quote["ask"] - mid) * config.order_size

        inventory_pnl = inventory_before * (effective_next_mid - mid)

        if abs(inventory) >= config.inventory_kill_limit:
            killed = True
            kill_reason = "INVENTORY_LIMIT"

        if mtm <= config.drawdown_kill_limit:
            killed = True
            kill_reason = "DRAWDOWN_LIMIT"

        rows.append({
            "step": step,
            "time_seconds": mid_df.loc[step, "time_seconds"],
            "strategy": strategy,
            "mid": mid,
            "next_mid": effective_next_mid,
            "inventory_before": inventory_before,
            "inventory_after": inventory,
            "cash_before": cash_before,
            "cash_after": cash,
            "mtm": mtm,
            "reservation_price": quote["reservation_price"],
            "bid": quote["bid"],
            "ask": quote["ask"],
            "spread": quote["spread"],
            "half_spread": quote["half_spread"],
            "bid_distance": quote["bid_distance"],
            "ask_distance": quote["ask_distance"],
            "lambda_bid": lambda_bid,
            "lambda_ask": lambda_ask,
            "p_bid": p_bid,
            "p_ask": p_ask,
            "bid_fill": int(bid_fill),
            "ask_fill": int(ask_fill),
            "spread_capture": spread_capture,
            "inventory_pnl": inventory_pnl,
            "killed": killed,
            "kill_reason": kill_reason,
        })

        if killed:
            break

    result = pd.DataFrame(rows)
    return result

path_as = run_market_making_path(config, strategy="AS", seed=config.seed)
path_sym = run_market_making_path(config, strategy="symmetric", seed=config.seed)

path_as.head()

## 8. Single-path diagnostics

Compare:

- inventory;
- quotes;
- PnL;
- fill counts;
- quote skew.

In [ ]:
def path_summary(path):
    if len(path) == 0:
        return {}

    nav = path["mtm"]
    dd = nav - nav.cummax()

    return {
        "strategy": path["strategy"].iloc[0],
        "n_steps": len(path),
        "final_mtm": path["mtm"].iloc[-1],
        "max_drawdown_dollars": dd.min(),
        "mean_inventory": path["inventory_after"].mean(),
        "max_abs_inventory": path["inventory_after"].abs().max(),
        "bid_fills": int(path["bid_fill"].sum()),
        "ask_fills": int(path["ask_fill"].sum()),
        "total_fills": int(path["bid_fill"].sum() + path["ask_fill"].sum()),
        "mean_spread": path["spread"].mean(),
        "mean_bid_distance": path["bid_distance"].mean(),
        "mean_ask_distance": path["ask_distance"].mean(),
        "killed": bool(path["killed"].iloc[-1]),
        "kill_reason": path["kill_reason"].iloc[-1],
    }

single_path_summary = pd.DataFrame([path_summary(path_as), path_summary(path_sym)])

single_path_summary

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(path_as["time_seconds"], path_as["inventory_after"], label="AS")
plt.plot(path_sym["time_seconds"], path_sym["inventory_after"], label="symmetric", alpha=0.75)
plt.axhline(0, linestyle="--")
plt.title("Inventory Path")
plt.xlabel("Time, seconds")
plt.ylabel("Inventory")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(path_as["time_seconds"], path_as["mtm"], label="AS")
plt.plot(path_sym["time_seconds"], path_sym["mtm"], label="symmetric", alpha=0.75)
plt.axhline(0, linestyle="--")
plt.title("Mark-to-Market PnL")
plt.xlabel("Time, seconds")
plt.ylabel("PnL")
plt.legend()
plt.show()

## 9. Quote skew diagnostics

Inventory-aware quoting should shift quotes:

- long inventory: lower reservation price;
- short inventory: higher reservation price.

We inspect the relation between inventory and quote skew.

Define:

$$
skew_t = \frac{bidDistance_t - askDistance_t}{spread_t}
$$

Positive skew means ask is closer than bid, encouraging sell fills.

Negative skew means bid is closer than ask, encouraging buy fills.

In [ ]:
path_as["quote_skew"] = (path_as["bid_distance"] - path_as["ask_distance"]) / path_as["spread"].replace(0, np.nan)
quote_skew_corr = path_as[["inventory_before", "quote_skew"]].corr().iloc[0, 1]

pd.Series({"inventory_quote_skew_corr": quote_skew_corr})

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(path_as["inventory_before"], path_as["quote_skew"], alpha=0.35)
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.title("Inventory vs Quote Skew")
plt.xlabel("Inventory before quote")
plt.ylabel("Quote skew")
plt.show()

## 10. Monte Carlo simulation

Run many paths for:

1. symmetric baseline;
2. Avellaneda-Stoikov.

Compare terminal PnL, inventory risk, fill counts, and kill-switch triggers.

In [ ]:
def run_monte_carlo(config, strategies=("symmetric", "AS"), volatility_multiplier=1.0, liquidity_multiplier=1.0, n_paths=None):
    if n_paths is None:
        n_paths = config.monte_carlo_paths

    summary_rows = []
    sample_paths = []

    for strategy in strategies:
        for path_id in range(n_paths):
            path = run_market_making_path(
                config,
                strategy=strategy,
                seed=config.seed + 10000 * (0 if strategy == "symmetric" else 1) + path_id,
                volatility_multiplier=volatility_multiplier,
                liquidity_multiplier=liquidity_multiplier,
            )
            summ = path_summary(path)
            summ["path_id"] = path_id
            summ["volatility_multiplier"] = volatility_multiplier
            summ["liquidity_multiplier"] = liquidity_multiplier
            summary_rows.append(summ)

            if path_id < 3:
                tmp = path.copy()
                tmp["path_id"] = path_id
                sample_paths.append(tmp)

    summary = pd.DataFrame(summary_rows)
    samples = pd.concat(sample_paths, ignore_index=True) if sample_paths else pd.DataFrame()

    return summary, samples

mc_summary, mc_sample_paths = run_monte_carlo(config, n_paths=config.monte_carlo_paths)

mc_summary.head()

## 11. Monte Carlo performance summary

In [ ]:
def monte_carlo_performance_summary(mc_summary):
    out = (
        mc_summary
        .groupby("strategy")
        .agg(
            n_paths=("path_id", "count"),
            mean_final_mtm=("final_mtm", "mean"),
            median_final_mtm=("final_mtm", "median"),
            std_final_mtm=("final_mtm", "std"),
            p05_final_mtm=("final_mtm", lambda x: x.quantile(0.05)),
            p95_final_mtm=("final_mtm", lambda x: x.quantile(0.95)),
            mean_max_drawdown=("max_drawdown_dollars", "mean"),
            p05_max_drawdown=("max_drawdown_dollars", lambda x: x.quantile(0.05)),
            mean_abs_inventory=("max_abs_inventory", "mean"),
            p95_abs_inventory=("max_abs_inventory", lambda x: x.quantile(0.95)),
            mean_total_fills=("total_fills", "mean"),
            kill_rate=("killed", "mean"),
        )
        .reset_index()
        .sort_values("mean_final_mtm", ascending=False)
    )
    return out

mc_performance = monte_carlo_performance_summary(mc_summary)

mc_performance

In [ ]:
plt.figure(figsize=(10, 5))
for strategy, group in mc_summary.groupby("strategy"):
    plt.hist(group["final_mtm"], bins=80, alpha=0.55, density=True, label=strategy)
plt.axvline(0, linestyle="--")
plt.title("Monte Carlo Terminal PnL")
plt.xlabel("Final MTM")
plt.ylabel("Density")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
for strategy, group in mc_summary.groupby("strategy"):
    plt.hist(group["max_abs_inventory"], bins=40, alpha=0.55, density=True, label=strategy)
plt.title("Monte Carlo Max Absolute Inventory")
plt.xlabel("Max absolute inventory")
plt.ylabel("Density")
plt.legend()
plt.show()

## 12. Volatility-regime stress test

The model should widen spreads and skew inventory under higher volatility.

We compare base and high-vol regimes.

In [ ]:
stress_base_summary, _ = run_monte_carlo(config, n_paths=config.sensitivity_paths, volatility_multiplier=1.0, liquidity_multiplier=1.0)
stress_highvol_summary, _ = run_monte_carlo(config, n_paths=config.sensitivity_paths, volatility_multiplier=2.0, liquidity_multiplier=1.0)

stress_base_summary["scenario"] = "base"
stress_highvol_summary["scenario"] = "high_vol"

vol_stress_summary = pd.concat([stress_base_summary, stress_highvol_summary], ignore_index=True)
vol_stress_perf = (
    vol_stress_summary
    .groupby(["scenario", "strategy"])
    .agg(
        mean_final_mtm=("final_mtm", "mean"),
        std_final_mtm=("final_mtm", "std"),
        p05_final_mtm=("final_mtm", lambda x: x.quantile(0.05)),
        mean_max_abs_inventory=("max_abs_inventory", "mean"),
        kill_rate=("killed", "mean"),
    )
    .reset_index()
)

vol_stress_perf

## 13. Liquidity-regime stress test

Lower liquidity reduces fill intensity.

This can reduce spread capture, but may also reduce inventory accumulation.

In [ ]:
liq_base_summary, _ = run_monte_carlo(config, n_paths=config.sensitivity_paths, volatility_multiplier=1.0, liquidity_multiplier=1.0)
liq_low_summary, _ = run_monte_carlo(config, n_paths=config.sensitivity_paths, volatility_multiplier=1.0, liquidity_multiplier=0.45)

liq_base_summary["scenario"] = "base_liquidity"
liq_low_summary["scenario"] = "low_liquidity"

liquidity_stress_summary = pd.concat([liq_base_summary, liq_low_summary], ignore_index=True)
liquidity_stress_perf = (
    liquidity_stress_summary
    .groupby(["scenario", "strategy"])
    .agg(
        mean_final_mtm=("final_mtm", "mean"),
        std_final_mtm=("final_mtm", "std"),
        mean_total_fills=("total_fills", "mean"),
        mean_max_abs_inventory=("max_abs_inventory", "mean"),
        kill_rate=("killed", "mean"),
    )
    .reset_index()
)

liquidity_stress_perf

## 14. Risk-aversion sensitivity

Risk aversion $\gamma$ controls inventory aversion and spread width.

Higher $\gamma$:

- widens quotes;
- shifts reservation price more aggressively;
- usually reduces inventory risk;
- may reduce fill rate and spread capture.

In [ ]:
def config_with_gamma(config, gamma):
    return AvellanedaStoikovConfig(
        horizon_seconds=config.horizon_seconds,
        dt_seconds=config.dt_seconds,
        seed=config.seed,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        order_size=config.order_size,
        initial_inventory=config.initial_inventory,
        max_inventory=config.max_inventory,
        gamma=gamma,
        sigma_daily=config.sigma_daily,
        trading_day_seconds=config.trading_day_seconds,
        arrival_A=config.arrival_A,
        arrival_k=config.arrival_k,
        taker_adverse_selection_bps=config.taker_adverse_selection_bps,
        min_spread_ticks=config.min_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        inventory_kill_limit=config.inventory_kill_limit,
        drawdown_kill_limit=config.drawdown_kill_limit,
        monte_carlo_paths=config.sensitivity_paths,
        sensitivity_paths=config.sensitivity_paths,
        output_subdir=config.output_subdir,
    )

gamma_grid = [0.005, 0.02, 0.05, 0.08, 0.15, 0.30]
gamma_rows = []

for gamma in gamma_grid:
    cfg = config_with_gamma(config, gamma)
    summ, _ = run_monte_carlo(cfg, strategies=("AS",), n_paths=config.sensitivity_paths)
    perf = monte_carlo_performance_summary(summ).iloc[0].to_dict()
    perf["gamma"] = gamma

    # Quote at q=10 as diagnostic.
    quote = make_quotes(config.initial_mid, inventory=10, step=0, n_steps=config.horizon_seconds, config=cfg, strategy="AS")
    perf["spread_at_q10"] = quote["spread"]
    perf["reservation_shift_at_q10"] = quote["reservation_price"] - config.initial_mid
    gamma_rows.append(perf)

gamma_sensitivity = pd.DataFrame(gamma_rows).sort_values("gamma")

gamma_sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gamma_sensitivity["gamma"], gamma_sensitivity["mean_final_mtm"], marker="o", label="mean PnL")
plt.xlabel("gamma")
plt.ylabel("Mean final MTM")
plt.title("PnL vs Risk Aversion")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(gamma_sensitivity["gamma"], gamma_sensitivity["p95_abs_inventory"], marker="o", label="p95 max abs inventory")
plt.xlabel("gamma")
plt.ylabel("Inventory")
plt.title("Inventory Risk vs Risk Aversion")
plt.legend()
plt.show()

## 15. Arrival sensitivity $k$

The parameter $k$ controls how quickly fill intensity decays as quotes move away from mid.

Higher $k$ means fills are very sensitive to distance:

- far quotes rarely fill;
- optimal liquidity component narrows;
- fill probability changes sharply with quote distance.

In [ ]:
def config_with_k(config, k):
    return AvellanedaStoikovConfig(
        horizon_seconds=config.horizon_seconds,
        dt_seconds=config.dt_seconds,
        seed=config.seed,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        order_size=config.order_size,
        initial_inventory=config.initial_inventory,
        max_inventory=config.max_inventory,
        gamma=config.gamma,
        sigma_daily=config.sigma_daily,
        trading_day_seconds=config.trading_day_seconds,
        arrival_A=config.arrival_A,
        arrival_k=k,
        taker_adverse_selection_bps=config.taker_adverse_selection_bps,
        min_spread_ticks=config.min_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        inventory_kill_limit=config.inventory_kill_limit,
        drawdown_kill_limit=config.drawdown_kill_limit,
        monte_carlo_paths=config.sensitivity_paths,
        sensitivity_paths=config.sensitivity_paths,
        output_subdir=config.output_subdir,
    )

k_grid = [0.40, 0.80, 1.20, 1.40, 2.00, 3.00]
k_rows = []

for k in k_grid:
    cfg = config_with_k(config, k)
    summ, _ = run_monte_carlo(cfg, strategies=("AS",), n_paths=config.sensitivity_paths)
    perf = monte_carlo_performance_summary(summ).iloc[0].to_dict()
    perf["arrival_k"] = k
    quote = make_quotes(config.initial_mid, inventory=0, step=0, n_steps=config.horizon_seconds, config=cfg, strategy="AS")
    perf["spread_at_q0"] = quote["spread"]
    perf["lambda_bid_at_q0"] = fill_intensity(quote["bid_distance"], cfg.arrival_A, cfg.arrival_k)
    k_rows.append(perf)

k_sensitivity = pd.DataFrame(k_rows).sort_values("arrival_k")

k_sensitivity

## 16. Adverse selection sensitivity

Market making earns spread, but fills are not always benign.

If informed traders hit your quotes, the mid-price may move against you after fills.

We vary the adverse-selection parameter.

In [ ]:
def config_with_adverse_selection(config, adverse_bps):
    return AvellanedaStoikovConfig(
        horizon_seconds=config.horizon_seconds,
        dt_seconds=config.dt_seconds,
        seed=config.seed,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        order_size=config.order_size,
        initial_inventory=config.initial_inventory,
        max_inventory=config.max_inventory,
        gamma=config.gamma,
        sigma_daily=config.sigma_daily,
        trading_day_seconds=config.trading_day_seconds,
        arrival_A=config.arrival_A,
        arrival_k=config.arrival_k,
        taker_adverse_selection_bps=adverse_bps,
        min_spread_ticks=config.min_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        inventory_kill_limit=config.inventory_kill_limit,
        drawdown_kill_limit=config.drawdown_kill_limit,
        monte_carlo_paths=config.sensitivity_paths,
        sensitivity_paths=config.sensitivity_paths,
        output_subdir=config.output_subdir,
    )

adverse_grid = [0.0, 0.05, 0.15, 0.30, 0.60, 1.00]
adverse_rows = []

for adverse in adverse_grid:
    cfg = config_with_adverse_selection(config, adverse)
    summ, _ = run_monte_carlo(cfg, strategies=("AS",), n_paths=config.sensitivity_paths)
    perf = monte_carlo_performance_summary(summ).iloc[0].to_dict()
    perf["adverse_selection_bps"] = adverse
    adverse_rows.append(perf)

adverse_sensitivity = pd.DataFrame(adverse_rows).sort_values("adverse_selection_bps")

adverse_sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(adverse_sensitivity["adverse_selection_bps"], adverse_sensitivity["mean_final_mtm"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Mean PnL vs Adverse Selection")
plt.xlabel("Adverse selection, bps")
plt.ylabel("Mean final MTM")
plt.show()

## 17. Quote quality diagnostics

We summarise quote properties:

- mean spread;
- spread distribution;
- mean quote distances;
- inventory skew;
- fill probabilities.

These diagnostics are crucial before interpreting PnL.

In [ ]:
quote_quality = pd.DataFrame([{
    "strategy": "AS",
    "mean_spread": path_as["spread"].mean(),
    "p95_spread": path_as["spread"].quantile(0.95),
    "mean_bid_distance": path_as["bid_distance"].mean(),
    "mean_ask_distance": path_as["ask_distance"].mean(),
    "mean_lambda_bid": path_as["lambda_bid"].mean(),
    "mean_lambda_ask": path_as["lambda_ask"].mean(),
    "mean_p_bid": path_as["p_bid"].mean(),
    "mean_p_ask": path_as["p_ask"].mean(),
    "inventory_quote_skew_corr": quote_skew_corr,
}])

quote_quality.T

## 18. Fill diagnostics

Fill diagnostics show whether the market maker is actually participating.

A market maker with no fills has no inventory risk, but also no business.

In [ ]:
fill_diagnostics = (
    path_as
    .agg(
        bid_fills=("bid_fill", "sum"),
        ask_fills=("ask_fill", "sum"),
        mean_p_bid=("p_bid", "mean"),
        mean_p_ask=("p_ask", "mean"),
        mean_lambda_bid=("lambda_bid", "mean"),
        mean_lambda_ask=("lambda_ask", "mean"),
        total_spread_capture=("spread_capture", "sum"),
        total_inventory_pnl=("inventory_pnl", "sum"),
    )
)

fill_diagnostics

## 19. Risk metrics

For a market-making strategy, risk is not just terminal PnL.

Important risk fields:

- inventory distribution;
- inventory limit hit rate;
- drawdown;
- kill-switch frequency;
- adverse selection sensitivity;
- PnL left tail.

In [ ]:
risk_report = pd.DataFrame([{
    "base_AS_mean_final_mtm": mc_performance.loc[mc_performance["strategy"] == "AS", "mean_final_mtm"].iloc[0],
    "base_AS_p05_final_mtm": mc_performance.loc[mc_performance["strategy"] == "AS", "p05_final_mtm"].iloc[0],
    "base_AS_p95_abs_inventory": mc_performance.loc[mc_performance["strategy"] == "AS", "p95_abs_inventory"].iloc[0],
    "base_AS_kill_rate": mc_performance.loc[mc_performance["strategy"] == "AS", "kill_rate"].iloc[0],
    "high_vol_AS_mean_final_mtm": vol_stress_perf[
        (vol_stress_perf["scenario"] == "high_vol") & (vol_stress_perf["strategy"] == "AS")
    ]["mean_final_mtm"].iloc[0],
    "low_liquidity_AS_mean_total_fills": liquidity_stress_perf[
        (liquidity_stress_perf["scenario"] == "low_liquidity") & (liquidity_stress_perf["strategy"] == "AS")
    ]["mean_total_fills"].iloc[0],
    "adverse_selection_break_even_near_bps": adverse_sensitivity.iloc[
        np.argmin(np.abs(adverse_sensitivity["mean_final_mtm"]))
    ]["adverse_selection_bps"],
}])

risk_report.T

## 20. Governance flags

A market-making model should be reviewed if:

1. inventory risk is too high;
2. kill-switch triggers too often;
3. PnL left tail is too negative;
4. model loses money under moderate adverse selection;
5. quote skew does not respond to inventory;
6. fill rate is too low;
7. volatility stress damages PnL too much;
8. parameters are synthetic / uncalibrated.

In [ ]:
as_perf = mc_performance[mc_performance["strategy"] == "AS"].iloc[0]
sym_perf = mc_performance[mc_performance["strategy"] == "symmetric"].iloc[0]

as_highvol = vol_stress_perf[
    (vol_stress_perf["scenario"] == "high_vol") & (vol_stress_perf["strategy"] == "AS")
].iloc[0]

as_lowliq = liquidity_stress_perf[
    (liquidity_stress_perf["scenario"] == "low_liquidity") & (liquidity_stress_perf["strategy"] == "AS")
].iloc[0]

adverse_030 = adverse_sensitivity[adverse_sensitivity["adverse_selection_bps"] == 0.30]["mean_final_mtm"].iloc[0]

governance_flags = pd.DataFrame([{
    "AS_mean_final_mtm": as_perf["mean_final_mtm"],
    "AS_p05_final_mtm": as_perf["p05_final_mtm"],
    "AS_p95_abs_inventory": as_perf["p95_abs_inventory"],
    "AS_kill_rate": as_perf["kill_rate"],
    "symmetric_mean_final_mtm": sym_perf["mean_final_mtm"],
    "inventory_quote_skew_corr": quote_skew_corr,
    "high_vol_mean_final_mtm": as_highvol["mean_final_mtm"],
    "low_liquidity_mean_total_fills": as_lowliq["mean_total_fills"],
    "adverse_030_mean_final_mtm": adverse_030,
    "flag_negative_mean_pnl": bool(as_perf["mean_final_mtm"] < 0),
    "flag_left_tail_loss_large": bool(as_perf["p05_final_mtm"] < -500.0),
    "flag_inventory_p95_near_limit": bool(as_perf["p95_abs_inventory"] > config.max_inventory * 0.90),
    "flag_kill_rate_high": bool(as_perf["kill_rate"] > 0.05),
    "flag_quote_skew_not_inventory_sensitive": bool(abs(quote_skew_corr) < 0.25),
    "flag_high_vol_pnl_negative": bool(as_highvol["mean_final_mtm"] < 0),
    "flag_low_liquidity_fills_too_low": bool(as_lowliq["mean_total_fills"] < as_perf["mean_total_fills"] * 0.50),
    "flag_adverse_selection_breaks_model": bool(adverse_030 < 0),
    "flag_parameters_uncalibrated": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_negative_mean_pnl",
        "flag_left_tail_loss_large",
        "flag_inventory_p95_near_limit",
        "flag_kill_rate_high",
        "flag_quote_skew_not_inventory_sensitive",
        "flag_high_vol_pnl_negative",
        "flag_low_liquidity_fills_too_low",
        "flag_adverse_selection_breaks_model",
        "flag_parameters_uncalibrated",
    ]
].any(axis=1)

governance_flags

## 21. Audit checks

Numerical and process checks:

1. bid is below ask;
2. fill probabilities are in $[0,1]$;
3. inventory updates are finite;
4. Monte Carlo outputs are finite;
5. risk sensitivity outputs are finite;
6. governance flags exist.

In [ ]:
audit_rows = []

bid_ask_ok = bool((path_as["bid"] < path_as["ask"]).all())
audit_rows.append({
    "check": "bid_less_than_ask",
    "value": bid_ask_ok,
    "passed": bid_ask_ok,
})

prob_ok = bool(
    ((path_as["p_bid"] >= 0) & (path_as["p_bid"] <= 1)).all()
    and ((path_as["p_ask"] >= 0) & (path_as["p_ask"] <= 1)).all()
)
audit_rows.append({
    "check": "fill_probabilities_in_unit_interval",
    "value": prob_ok,
    "passed": prob_ok,
})

inventory_finite = bool(np.isfinite(path_as[["inventory_before", "inventory_after", "cash_after", "mtm"]].to_numpy()).all())
audit_rows.append({
    "check": "single_path_inventory_cash_finite",
    "value": inventory_finite,
    "passed": inventory_finite,
})

mc_finite = bool(np.isfinite(mc_summary.select_dtypes(include=[float, int]).to_numpy()).all())
audit_rows.append({
    "check": "monte_carlo_outputs_finite",
    "value": mc_finite,
    "passed": mc_finite,
})

sensitivity_finite = bool(
    np.isfinite(gamma_sensitivity.select_dtypes(include=[float, int]).to_numpy()).all()
    and np.isfinite(k_sensitivity.select_dtypes(include=[float, int]).to_numpy()).all()
    and np.isfinite(adverse_sensitivity.select_dtypes(include=[float, int]).to_numpy()).all()
)
audit_rows.append({
    "check": "sensitivity_outputs_finite",
    "value": sensitivity_finite,
    "passed": sensitivity_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 22. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

mid_path.to_csv(output_dir / "mid_price_path.csv", index=False)
path_as.to_csv(output_dir / "single_path_avellaneda_stoikov.csv", index=False)
path_sym.to_csv(output_dir / "single_path_symmetric.csv", index=False)
single_path_summary.to_csv(output_dir / "single_path_summary.csv", index=False)
quote_quality.to_csv(output_dir / "quote_quality.csv", index=False)
fill_diagnostics.to_frame("value").to_csv(output_dir / "fill_diagnostics.csv")
mc_summary.to_csv(output_dir / "monte_carlo_paths_summary.csv", index=False)
mc_performance.to_csv(output_dir / "monte_carlo_performance.csv", index=False)
mc_sample_paths.to_csv(output_dir / "monte_carlo_sample_paths.csv", index=False)
vol_stress_perf.to_csv(output_dir / "volatility_stress_performance.csv", index=False)
liquidity_stress_perf.to_csv(output_dir / "liquidity_stress_performance.csv", index=False)
gamma_sensitivity.to_csv(output_dir / "gamma_sensitivity.csv", index=False)
k_sensitivity.to_csv(output_dir / "arrival_k_sensitivity.csv", index=False)
adverse_sensitivity.to_csv(output_dir / "adverse_selection_sensitivity.csv", index=False)
risk_report.to_csv(output_dir / "risk_report.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "avellaneda_stoikov_market_making_outputs",
    "schema_version": "avellaneda_stoikov_market_making_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "single_path_summary": single_path_summary.to_dict(orient="records"),
    "monte_carlo_performance": mc_performance.to_dict(orient="records"),
    "quote_quality": quote_quality.to_dict(orient="records"),
    "risk_report": risk_report.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook implements a simplified Avellaneda-Stoikov market-making simulator.",
        "Quotes are generated from reservation price and optimal half-spread formulas.",
        "Fill arrivals follow exponential quote-distance intensity.",
        "Inventory, cash, spread capture, inventory PnL and MTM are tracked.",
        "Symmetric and inventory-aware market makers are compared.",
        "Volatility, liquidity, risk-aversion, arrival-k and adverse-selection sensitivity are reported.",
        "Parameters are illustrative and must be calibrated from real L1/order/fill data before production."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "monte_carlo_performance.csv", output_dir / "gamma_sensitivity.csv", output_dir / "governance_flags.csv", manifest_path

## 23. Practical checklist for Avellaneda-Stoikov market making

Before using this model:

1. **Calibrate fill intensity $A,k$** from quote-distance and fill data.
2. **Estimate volatility at the quoting horizon.**
3. **Choose risk aversion $\gamma$** from inventory risk tolerance.
4. **Respect tick size and minimum spread.**
5. **Add inventory limits and kill switches.**
6. **Model adverse selection.**
7. **Stress volatility and liquidity regimes.**
8. **Compare against symmetric quoting.**
9. **Validate quote skew versus inventory.**
10. **Backtest with L1 bid/ask and realistic fill assumptions.**

## 24. Limitations

### Simplified fill model

Fills are independent Bernoulli events generated from quote-distance intensity. Real fills depend on queue position, order size, cancellations, hidden liquidity and competing quotes.

### No full order book

The notebook does not model L2 depth or price-time priority.

### Simplified adverse selection

Adverse selection is modelled as a fixed post-fill move. Real adverse selection is state-dependent.

### No latency

Quotes are assumed to update instantly. Production market making requires latency and stale-quote risk.

### No exchange constraints

No price limits, margin, throttles or order-to-trade ratio rules are included here.

### Synthetic parameters

The model uses illustrative parameters. Production requires calibration from quote/fill/order data.

## 25. Research frontier and extensions

Important extensions include:

1. queue-reactive market making;
2. microprice-aware quoting;
3. Hawkes order-flow intensities;
4. latency-aware stale quote risk;
5. multi-asset market making;
6. adverse-selection classifiers;
7. reinforcement-learning market making;
8. inventory-aware skew with hard constraints;
9. exchange-specific throttles and cancel rules;
10. futures market making with margin and price-limit boards.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_09_execution_risk_controls_and_kill_switch**  
   Add hard safety controls around market making.

2. **06_10_futures_roll_execution_engine**  
   Handle futures-specific rolling and liquidity transitions.

3. **06_11_l1_bid_ask_backtest_execution_model**  
   Test market-making quotes on L1 data.

4. **06_12_production_reconciliation_and_breaks**  
   Reconcile quoted, acknowledged, filled and cancelled orders.

5. **06_13_execution_research_capstone**  
   Combine execution, market making, impact and risk controls.

## 27. Summary

This notebook implemented Avellaneda-Stoikov market making.

It showed how to:

1. define reservation price;
2. compute optimal half-spread;
3. generate inventory-aware quotes;
4. model quote-distance fill intensities;
5. simulate bid and ask fills;
6. track inventory and cash;
7. decompose spread capture and inventory PnL;
8. compare symmetric and AS market making;
9. run Monte Carlo PnL simulations;
10. stress volatility and liquidity;
11. analyse risk-aversion sensitivity;
12. analyse fill-intensity sensitivity;
13. analyse adverse-selection sensitivity;
14. build quote and fill diagnostics;
15. create governance flags and audit checks;
16. save outputs and manifests.

The key computational takeaway:

> Avellaneda-Stoikov market making converts inventory risk into quote skew through the reservation price.

The key financial takeaway:

> Market making is not just earning spread; it is managing the inventory you accidentally acquire while trying to earn that spread.

## 28. Further reading

- Avellaneda and Stoikov, "High-frequency trading in a limit order book."
- Guéant, Lehalle and Fernandez-Tapia on dealing with inventory risk.
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Abergel et al., *Limit Order Books.*
- Fodra and Labadie on high-frequency market making.
- Guéant, *The Financial Mathematics of Market Liquidity.*